In [ ]:
import numpy as np
import pandas as pd
import os
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
import torch
from tqdm import tqdm
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import ast
import re
from peft import PeftModel

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# MODEL

In [ ]:
# MODEL_ID = 'Kiffaz11/qwen2.5-3b-absa-lora'
# eval_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# base_model = AutoModelForCausalLM.from_pretrained(
#     'Qwen/Qwen2.5-3B-Instruct',
#     torch_dtype="auto",
#     device_map='balanced',
# )
# pre_eval_model = PeftModel.from_pretrained(base_model, MODEL_ID)
# eval_model = pre_eval_model.merge_and_unload()


MODEL_ID = 'vohuutridung/vit5-large-absa'
eval_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
eval_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map='auto',
)

eval_model.eval()

In [ ]:
@torch.no_grad()
def generate_response_causallm(reviews, tokenizer, model,):
    SYSTEM_PROMPT = (
        "Bạn là hệ thống trích xuất khía cạnh và cảm xúc (Aspect-Based Sentiment Analysis) cho review sản phẩm.\n"
        "Nhiệm vụ: từ một đoạn review tiếng Việt, trích xuất các mục theo định dạng JSON:\n"
        '[[ "aspect_term", "aspect_category", "sentiment", "opinion_phrase" ], ...]\n'
        "Quy tắc:\n"
        "- Chỉ trả về JSON thuần (không giải thích, không markdown).\n"
        "- Giữ nguyên nhãn như dữ liệu (ví dụ: TỔNG_QUAN, PIN, ...; TÍCH_CỰC/TIÊU_CỰC/TRUNG_LẬP).\n"
        '- Nếu thiếu aspect_term hoặc opinion_phrase, dùng chuỗi "NULL" đúng như dữ liệu.\n'
    )
    
    prompts = []
    for review in reviews:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": review.strip()},
        ]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt)

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024,
    ).to(device)

    input_lens = inputs["attention_mask"].sum(dim=1)

    gen_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        num_beams=1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    outputs = []
    for i in range(len(prompts)):
        gen_part = gen_ids[i][int(input_lens[i]):]
        out_text = tokenizer.decode(gen_part, skip_special_tokens=True)
        outputs.append(out_text)
    return outputs
    

@torch.no_grad()
def generate_response_seq2seqlm(reviews, tokenizer, model,):
    inputs = tokenizer(
        reviews,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_length=256,
    )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# DATA

In [ ]:
eval_dataset = load_dataset('vohuutridung/3190-data', split='test')
eval_dataset

# SCORE

In [ ]:
QUAD_RE = re.compile(
    r'\[\s*'
    r'"([^"\n\r]*)"\s*,\s*'
    r'"([^"\n\r]*)"\s*,\s*'
    r'"([^"\n\r]*)"\s*,\s*'
    r'"([^"\n\r]*)"\s*'
    r'\]',
    re.UNICODE
)

def parse_causallm(raw_text):
    if not raw_text:
        return []

    s = str(raw_text).strip()
    results = []

    for parser in (json.loads, ast.literal_eval):
        try:
            obj = parser(s)
            if isinstance(obj, list):
                if all(isinstance(x, list) and len(x) == 4 for x in obj):
                    return obj
                if len(obj) == 4 and all(isinstance(x, str) for x in obj):
                    return [obj]
        except Exception:
            pass

    try:
        m = re.search(r'\[\s*\[.*?\]\s*\]', s, re.DOTALL)
        if m:
            obj = json.loads(m.group())
            if isinstance(obj, list):
                return obj
    except Exception:
        pass

    quads = QUAD_RE.findall(s)
    for q in quads:
        if len(q) == 4:
            results.append(list(q))

    return results

In [ ]:
CATEGORIES = {
    "TỔNG_QUAN","PIN","HIỆU_NĂNG","MÁY_ẢNH","MÀN_HÌNH",
    "GIÁ_CẢ","TÍNH_NĂNG","THIẾT_KẾ","DỊCH_VỤ&PHỤ_KIỆN","LƯU TRỮ"
}
SENTIMENT = {"TÍCH_CỰC","TIÊU_CỰC","TRUNG_LẬP"}
QUAD_RE = re.compile(r"\[\s*'([^']*)'\s*,\s*'([^']*)'\s*,\s*'([^']*)'\s*,\s*'([^']*)'\s*\]")

def parse_seq2seqlm(raw_text):
    """
    Robust parser for ViT5 / seq2seq ABSA output.
    - Ignores broken / truncated quadruples
    - Extracts only fully-formed quadruples
    - NEVER hallucinates fields
    """
    if not raw_text:
        return []

    s = str(raw_text)

    results = []
    for a, c, se, o in QUAD_RE.findall(s):
        a, c, se, o = a.strip(), c.strip(), se.strip(), o.strip()

        # Hard validation (quan trọng)
        if c not in CATEGORIES:
            continue
        if se not in SENTIMENT:
            continue

        results.append([a, c, se, o])

    return results

In [ ]:
def _norm_text(x):
    x = "" if x is None else str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def _norm_label(x):
    return _norm_text(x).upper()

def to_sets(quads):
    """
    Convert list-of-lists -> normalized sets:
    - ac_set: (aspect_term, category)
    - tri_set: (aspect_term, category, sentiment)
    - quad_set: (aspect_term, category, sentiment, opinion_phrase)
    """
    ac_set, tri_set, quad_set = set(), set(), set()
    if not isinstance(quads, list):
        return ac_set, tri_set, quad_set

    for q in quads:
        if not isinstance(q, (list, tuple)) or len(q) < 4:
            continue
        a, c, s, o = q[0], q[1], q[2], q[3]
        a_n = _norm_text(a)
        c_n = _norm_label(c)
        s_n = _norm_label(s)
        o_n = _norm_text(o)

        ac_set.add((a_n, c_n))
        tri_set.add((a_n, c_n, s_n))
        quad_set.add((a_n, c_n, s_n, o_n))
    return ac_set, tri_set, quad_set

def prf(overlap, pred_total, ref_total):
    p = overlap / pred_total if pred_total > 0 else 0.0
    r = overlap / ref_total if ref_total > 0 else 0.0
    f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
    return p, r, f1

In [ ]:
bs = 64
responses = []
predictions = []
references = []

parse_miss = 0

if eval_model.config.is_encoder_decoder:
    for i in tqdm(range(0, len(eval_dataset), bs), desc="Generating"):
        batch = eval_dataset[i : i + bs]
        preds_text = generate_response_seq2seqlm(batch["text"], eval_tokenizer, eval_model)
        preds_parsed = []
        for x in preds_text:
            xx = parse_seq2seqlm(x)
            if len(xx) == 0: parse_miss += 1
            preds_parsed.append(xx)
            
        responses.extend(preds_text)
        predictions.extend(preds_parsed)
        references.extend(batch["labels"])
else:
    for i in tqdm(range(0, len(eval_dataset), bs), desc="Generating"):
        batch = eval_dataset[i : i + bs]
        preds_text = generate_response_causallm(batch["text"], eval_tokenizer, eval_model)
        preds_parsed = []
        for x in preds_text:
            xx = parse_causallm(x)
            if len(xx) == 0: parse_miss += 1
            preds_parsed.append(xx)

        responses.extend(preds_text)
        predictions.extend(preds_parsed)
        references.extend(batch["labels"])

In [ ]:
# ---- Compute metrics ----
ac_overlap = ac_pred_total = ac_ref_total = 0
tri_overlap = tri_pred_total = tri_ref_total = 0
quad_overlap = quad_pred_total = quad_ref_total = 0

exact_match_count = 0

sent_correct = 0
sent_total = 0

for pred, ref in zip(predictions, references):
    pred_ac, pred_tri, pred_quad = to_sets(pred)
    ref_ac, ref_tri, ref_quad = to_sets(ref)

    ac_overlap += len(pred_ac & ref_ac)
    ac_pred_total += len(pred_ac)
    ac_ref_total += len(ref_ac)

    tri_overlap += len(pred_tri & ref_tri)
    tri_pred_total += len(pred_tri)
    tri_ref_total += len(ref_tri)

    quad_overlap += len(pred_quad & ref_quad)
    quad_pred_total += len(pred_quad)
    quad_ref_total += len(ref_quad)

    if pred_quad == ref_quad:
        exact_match_count += 1

    # Sentiment accuracy on matched (aspect_term, category)
    # Build dict from (a,c) -> sentiment
    pred_map = {}
    for (a, c, s) in pred_tri:
        pred_map[(a, c)] = s
    ref_map = {}
    for (a, c, s) in ref_tri:
        ref_map[(a, c)] = s

    matched_keys = set(pred_map.keys()) & set(ref_map.keys())
    for k in matched_keys:
        sent_total += 1
        if pred_map[k] == ref_map[k]:
            sent_correct += 1

ac_p, ac_r, ac_f1 = prf(ac_overlap, ac_pred_total, ac_ref_total)
tri_p, tri_r, tri_f1 = prf(tri_overlap, tri_pred_total, tri_ref_total)
quad_p, quad_r, quad_f1 = prf(quad_overlap, quad_pred_total, quad_ref_total)

exact_match = exact_match_count / len(references) if len(references) else 0.0
sent_acc = sent_correct / sent_total if sent_total else 0.0

metrics = {
    "aspect_category_micro_precision": ac_p,
    "aspect_category_micro_recall": ac_r,
    "aspect_category_micro_f1": ac_f1,

    "triplet_micro_precision": tri_p,
    "triplet_micro_recall": tri_r,
    "triplet_micro_f1": tri_f1,

    "quadruple_micro_precision": quad_p,
    "quadruple_micro_recall": quad_r,
    "quadruple_micro_f1": quad_f1,

    "exact_match_rate": exact_match,
    "sentiment_accuracy_on_matched_aspects": sent_acc,

    "counts": {
        "ac_pred_total": ac_pred_total,
        "ac_ref_total": ac_ref_total,
        "tri_pred_total": tri_pred_total,
        "tri_ref_total": tri_ref_total,
        "quad_pred_total": quad_pred_total,
        "quad_ref_total": quad_ref_total,
        "sent_matched_total": sent_total,
    },
    "parse_miss": parse_miss,
}

print(json.dumps(metrics, indent=2, ensure_ascii=False))